## OpenAI Connection

In [ ]:
#%run /home/chander/code/github/dbx-genai-repo/Course2/.setup/learner_setup.ipynb

In [ ]:
import openai
import os
import json
import httpx
from dotenv import load_dotenv
from tenacity import (
    retry,
    stop_after_attempt,
    wait_random_exponential,
)

In [ ]:
# Load variables from .env file
load_dotenv()
# Access variables
api_key = os.getenv('OPENAI_API_KEY')
base_url = os.getenv('OPENAI_BASE_URL')
chat_model_name = os.getenv('CHAT_MODEL_NAME')
embedding_model_name = os.getenv('EMBEDDING_MODEL_NAME')
chat_client = openai.OpenAI(
    api_key=api_key,
    base_url=base_url
)
embedding_client = openai.OpenAI(
    api_key=api_key,
    base_url=base_url
)

In [ ]:
# We use an exponential backoff decorator in the event too 
# many users are using the model at once and rate limit is reached
@retry(wait=wait_random_exponential(min=45, max=120), stop=stop_after_attempt(6))
def query_llm(prompt_messages, max_tokens=4096, temperature=1.0, top_p=1.0):
    
    response = chat_client.chat.completions.create(
        messages=prompt_messages,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        model=chat_model_name
    )

    return {'text': response.choices[0].message.content}

In [ ]:
query = "List top 5 commonly used classification models used in machine learning"
prompt_messages = [
    {"role": "user", "content": query}
]

response = query_llm(prompt_messages)
print(response['text'])

In [ ]:
sample_text = """
Reasoned about discharge summary sample for 11 seconds
Patient admitted with community-acquired pneumonia received IV antibiotics with marked clinical improvement.
Fever resolved and oxygenation normalized over a 5-day hospital stay.
Discharge medications include oral azithromycin and supportive care instructions.
Follow-up is scheduled in 1 week to reassess recovery.
"""
prompt_messages=[
        {"role": "developer", "content": "You will be provided with a patient discharge summary, and your task is to extract keywords from it"},
        {"role": "user", "content": f"Extract keywords from this patient discharge summary:{sample_text}"},  
    ]

response = query_llm(prompt_messages)
print(response['text'])


In [ ]:
response = embedding_client.embeddings.create(
    input=["hello from litellm"],
    model=embedding_model_name
)

print(response)